# Final configuration (sqrt 4-class, imgsz 1024 rect): Wise-IoU v3 ablation and seed repeats
1. **Wise-IoU v3** box loss (train_wiou.py) instead of CIoU, everything else identical (seed 42).
2. The unchanged final configuration with **seeds 1 and 2** (the seed-42 run already exists, val mAP50 75.3), to measure run-to-run spread.

**Pre-registered rule (fixed before any result):** Wise-IoU replaces CIoU in the final model only if its 4-class val mAP50 exceeds the
seed-42 CIoU run by more than the seed spread (max minus min val mAP50 over seeds 42, 1, 2). If it does, it is evaluated on test once.
The seed runs are reported on val only (no test evaluation).

In [ ]:
!pip -q install ultralytics==8.4.142
!nvidia-smi --query-gpu=name,memory.total --format=csv
import ultralytics, torch, os, glob, shutil, time, json, subprocess, sys
T0 = time.time()
yml = glob.glob('/kaggle/input/**/dataset_clean/data.yaml', recursive=True); assert yml
SRC = os.path.dirname(yml[0])
hits = glob.glob('/kaggle/input/**/train_wiou.py', recursive=True)
assert hits, 'train_wiou.py not found - is wood-defects-scripts (v5+) attached?'
SCRIPTS = os.path.dirname(hits[0]); print('scripts dir:', SCRIPTS, os.listdir(SCRIPTS))
DATA = '/kaggle/tmp/dataset_clean'
if not os.path.isdir(DATA + '/train/images'):
    shutil.copytree(SRC, DATA)
OUT = '/kaggle/tmp/dataset_4cls'
rc = subprocess.call([sys.executable, f'{SCRIPTS}/build_balanced.py', '--src', DATA, '--out', OUT, '--seed', '42'])
assert rc == 0
V1 = ['train_sqrt_042fff_104900047s0216.jpg', 'train_sqrt_24b8d1_111200005s0176.jpg', 'train_sqrt_5416f5_111200047s0174.jpg',
      'train_sqrt_5afb6d_106100037s0357.jpg', 'train_sqrt_c826f2_109300034s0179.jpg', 'train_sqrt_e72b6c_100500037s0412.jpg']
SAME = sorted(f for f in os.listdir(f'{OUT}/previews') if f.startswith('train_sqrt_')) == V1
print('identical build to v1:', SAME)

In [ ]:
VARIANT, IMGSZ = 'sqrt', 1024
def train(script, name, seed, batch=32):
    cmd = [sys.executable, f'{SCRIPTS}/{script}', '--data', f'{OUT}/data_{VARIANT}.yaml', '--name', name, '--device', '0',
           '--model', 'yolo11s.pt', '--batch', str(batch), '--lr0', '0.001', '--patience', '40', '--imgsz', str(IMGSZ),
           '--project', '/kaggle/working/outputs', '--persist', '/kaggle/working/persist', '--seed', str(seed)]
    print(' '.join(cmd)); t = time.time(); rc = subprocess.call(cmd)
    ok = os.path.exists(f'/kaggle/working/outputs/{name}/weights/best.pt')
    print(name, 'exit', rc, 'OK' if ok else 'FAILED', f'{(time.time()-t)/3600:.2f} h'); return ok

RUNS = [('train_wiou.py', f'yolo11s_4cls_{VARIANT}_{IMGSZ}_wiou', 42),
        ('train_yolo11m.py', f'yolo11s_4cls_{VARIANT}_{IMGSZ}_seed1', 1),
        ('train_yolo11m.py', f'yolo11s_4cls_{VARIANT}_{IMGSZ}_seed2', 2)]
done = []
for script, name, seed in RUNS:
    if (time.time() - T0) / 3600 > 8.5:
        print('SKIP', name); continue
    if train(script, name, seed) or train(script, name + '_b16', seed, 16):
        done.append(name if os.path.exists(f'/kaggle/working/outputs/{name}/weights/best.pt') else name + '_b16')
print(done)

In [ ]:
val = {}
for name in done:
    out = f'/kaggle/working/eval/val_{name}'
    subprocess.call([sys.executable, f'{SCRIPTS}/evaluate.py', '--weights', f'/kaggle/working/outputs/{name}/weights/best.pt',
                     '--data', f'{OUT}/data_{VARIANT}.yaml', '--split', 'val', '--imgsz', str(IMGSZ), '--device', '0', '--out', out])
    md = open(f'{out}/val_metrics.md').read(); print(name); print(md)
    val[name] = float([l.split('|') for l in md.splitlines() if l.startswith('| ALL')][0][5])
seeds = [75.3] + [v for k, v in val.items() if 'seed' in k]
spread = max(seeds) - min(seeds)
w = [v for k, v in val.items() if 'wiou' in k]
ADOPT = bool(w) and len(seeds) == 3 and w[0] - 75.3 > spread
json.dump(dict(val_map50=val, seed_val_map50=seeds, seed_spread=spread, wiou_adopted=ADOPT), open('/kaggle/working/selection.json', 'w'), indent=2)
print('seed val mAP50', seeds, 'spread', round(spread, 2), '| WIoU', w, '-> adopted:', ADOPT)

In [ ]:
if ADOPT:
    name = [k for k in done if 'wiou' in k][0]; out = f'/kaggle/working/eval/test_{name}'
    subprocess.call([sys.executable, f'{SCRIPTS}/evaluate.py', '--weights', f'/kaggle/working/outputs/{name}/weights/best.pt',
                     '--data', f'{OUT}/data_{VARIANT}.yaml', '--split', 'test', '--imgsz', str(IMGSZ), '--device', '0', '--out', out])
    print(open(f'{out}/test_metrics.md').read())
print(f'total wall-clock {(time.time() - T0) / 3600:.2f} h')